# Model Compression with LLM Compressor

Compress a Granite language model using INT8 quantization and compare file sizes before and after compression.

**Model:** IBM Granite 4.0 350M (350M parameters)  
**Quantization:** INT8 W8A16 (weight-only)  

**Prerequisites:**
- Workbench: **Jupyter | PyTorch LLM Compressor | CUDA | Python 3.12**
- Shared PVC mounted at `/shared-models` with Granite model

### 1. Import required libraries.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier
from accelerate import dispatch_model
from accelerate.utils.modeling import infer_auto_device_map
from pathlib import Path

print("✓ Libraries imported")

### 2. Configure model paths.

In [ ]:
# Model is pre-loaded from OCI image to shared PVC
MODEL_PATH = "/shared-models/granite-4.0-350m"
SAVE_DIR = "granite-INT8"

print(f"Model path: {MODEL_PATH}")
print(f"Output directory: {SAVE_DIR}")

### 3. Load the original model from shared storage.

In [ ]:
# Verify model path exists
import os
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model path not found: {MODEL_PATH}")

# Check for required model files
required_files = ["config.json", "tokenizer_config.json"]
for file in required_files:
    file_path = os.path.join(MODEL_PATH, file)
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Required file not found: {file_path}")

print(f"✓ Model path verified: {MODEL_PATH}")
print(f"  Files: {', '.join(os.listdir(MODEL_PATH))}")

# Load model from shared storage (pre-extracted from OCI image)
# Use absolute path to ensure transformers recognizes it as local
model = AutoModelForCausalLM.from_pretrained(
    os.path.abspath(MODEL_PATH),
    torch_dtype="auto",
    low_cpu_mem_usage=True,
    local_files_only=True,
    trust_remote_code=False
)
tokenizer = AutoTokenizer.from_pretrained(
    os.path.abspath(MODEL_PATH),
    local_files_only=True,
    trust_remote_code=False
)

print("✓ Model loaded from shared storage")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

### 4. Test generation with the original model.

In [ ]:
test_prompt = "Hello, my name is"
inputs = tokenizer(test_prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=20, do_sample=False)
original_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Original output: '{original_text}'")

### 5. Configure and apply INT8 quantization.

In [ ]:
recipe = QuantizationModifier(
    targets="Linear",
    scheme="W8A16",      # INT8 weights, FP16 activations
    ignore=[]
)

print("Applying INT8 quantization...")
oneshot(model=model, recipe=recipe)
print("✓ Quantization complete")

### 6. Test generation with the quantized model.

In [ ]:
device_map = infer_auto_device_map(model)
model = dispatch_model(model, device_map)

inputs = tokenizer(test_prompt, return_tensors="pt")
input_ids = inputs.input_ids.to(model.device)
outputs = model.generate(input_ids, max_new_tokens=20, do_sample=False)
quantized_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Quantized output: '{quantized_text}'")

### 7. Save the compressed model.

In [ ]:
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"✓ Model saved to {SAVE_DIR}/")

### 8. Compare file sizes.

In [ ]:
import subprocess

def get_size_mb(path):
    result = subprocess.run(["du", "-sm", path], capture_output=True, text=True)
    return int(result.stdout.split()[0])

# Get sizes
original_size = get_size_mb(MODEL_PATH)
compressed_size = get_size_mb(SAVE_DIR)
reduction = ((original_size - compressed_size) / original_size) * 100

print("="*50)
print(f"Original:    {original_size:>6} MB")
print(f"Compressed:  {compressed_size:>6} MB")
print(f"Reduction:   {reduction:>6.1f}%")
print("="*50)

### 9. Summary.

In [ ]:
print("\n✓ Model compressed successfully")
print(f"✓ File size reduced by ~{reduction:.0f}%")
print(f"✓ Saved to: {SAVE_DIR}/")